# Analysis — PPO from scratch on Snake

This notebook loads the trained checkpoint, evaluates it against a random baseline, and (optionally) compares it with the Stable-Baselines3 PPO baseline.

Run training first: `python -m training.train --config training/configs/snake_ppo.yaml`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
from envs.snake_env import SnakeEnv
from agents.ppo_scratch import PPO
from evaluation.evaluate import run_policy

## 1. Random baseline vs PPO from scratch

In [ ]:
env = SnakeEnv(grid_size=12)
rng = np.random.default_rng(0)
random_stats = run_policy(lambda o: int(rng.integers(env.action_space.n)), env, episodes=100)

agent = PPO.load('../checkpoints/best_model.pt')
ppo_stats = run_policy(agent.act_greedy, env, episodes=100)

print('random:', random_stats)
print('ppo   :', ppo_stats)

## 2. Learning curve

Rendered from the TensorBoard logs (see `evaluation/plot_curves.py`).

In [ ]:
from evaluation.plot_curves import load_scalars
import matplotlib.pyplot as plt

steps, reward = load_scalars('../runs/snake_ppo_main', 'rollout/mean_ep_reward')
plt.figure(figsize=(8, 4))
plt.plot(steps, reward)
plt.xlabel('environment steps'); plt.ylabel('mean episode reward')
plt.title('PPO-from-scratch learning curve'); plt.grid(alpha=0.3)
plt.show()

## 3. Score distribution over 100 evaluation episodes

In [ ]:
scores = []
for i in range(100):
    obs, _ = env.reset(seed=i)
    done = False
    while not done:
        obs, _, term, trunc, info = env.step(agent.act_greedy(obs))
        done = term or trunc
    scores.append(info['score'])

plt.figure(figsize=(7, 4))
plt.hist(scores, bins=range(0, max(scores) + 2))
plt.xlabel('game score (food eaten)'); plt.ylabel('episodes')
plt.title(f'Score distribution — mean {np.mean(scores):.2f}, max {max(scores)}')
plt.show()